# <div align="center"> Statistical learning </div>
# <div align="center"> Machine Learning and Econometrics  </div>
## <div align="center"> ECO 4971/6971  </div>
#### <div align="center">Class 11 — Tree-based methods</div>
<div align="center"> Jonathan Holmes (he/him)</div>


# Road map

We use the usual framework: predict $y$ from predictors $X$ in
$$y = f(X) + \varepsilon.$$
So far, $f$ has often been **linear** in $X$. Today, $f$ is built from **trees**: nested **if–then** splits that partition the predictor space into **regions** (rectangles when splits are axis-aligned).

**Learning objectives.** By the end of this class you should be able to:
- Read a **regression** or **classification** tree and connect branches to **regions** and **predictions** (means or majority vote).
- Explain how trees are grown by **greedy** optimization (RSS for regression; Gini or entropy for classification).
- Describe **bagging**, **random forests**, and **boosting**, and what problem each is meant to address (especially **variance** vs. **bias**).
- Interpret **out-of-bag (OOB)** error, **feature importance**, and basic boosting knobs ($B$, tree depth, learning rate).

**Plan for today:** (1) Regression trees — *Hitters* demo → (2) Classification trees — *Heart* → (3) Trees vs. linear models → (4) Ensembles → (5) Case study — *Boston*.

**In-class work:** Questions **Q1–Q11** appear below in six blocks (**In-Class Exercise #1–#6**), in a single numbered list from the top of the notebook.


# Part 1: Decision trees for regression

A **decision tree** predicts $y$ by sending each observation down a path of **splits** until it lands in a **terminal node (leaf)**. In regression, every observation in the same leaf gets the **same** prediction: the **mean** of $y$ among training points in that leaf (here, the mean of **log salary**).

This is a **nonparametric** partition of the feature space: we do not estimate a $\beta$ vector for the whole space at once—we carve space into **boxes** and fit a simple constant in each box.


## Motivating example: Major League Baseball salaries

We use the **Hitters** data (classic example from *Introduction to Statistical Learning*). You want to **predict** a player's salary from experience and performance.

**Variables for this demo:**
- **Salary** (thousands of dollars) — we model **log salary** after cleaning.
- **Years** — years in the major leagues.
- **Hits** — hits in the previous season.

**Cleaning:** drop rows with missing salary; take **log(Salary)** so the outcome is less skewed (similar spirit to other chapters in this course).

*As you run the code cells, ask: What is the fitted tree doing that a single linear equation in Years and Hits would not?*


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv('https://www.dropbox.com/scl/fi/0sdjzynr4369663yhczrg/Hitters.csv?rlkey=zme3dppn0e21vrrsvzpxa2vwh&dl=1')
df.pop(df.columns[0])
print("Before Data Cleaning")
display(df.head())
df=df.loc[:,['Salary','Years','Hits']] # keep only the variables needed
df=df.loc[~df['Salary'].isna()] # keep only if observations have information on salary (i.e. if Salary is not (~) missing)
df['salary']=np.log(df['Salary']) # create log of Salary and assign to salary (small cap)
display(df.describe().T)
print("After Data Cleaning")
df.head()

In [ ]:
fig, axes =plt.subplots(1,2, figsize=(12,8))

sns.histplot(x=df['Salary'], stat='frequency', kde=True,color='darkorange', fill=False,line_kws={'color':'k'}, ax=axes[0])
sns.histplot(x=df['salary'], stat='density', kde=True,color='darkgreen', fill=False, ax=axes[1])

axes[0].set_xlabel("Salary in thousands",fontsize=12)
axes[1].set_xlabel("Log-Transform of Salary",fontsize=12)
axes[0].set_ylabel("Frequency",fontsize=12)
axes[1].set_ylabel("Frequency",fontsize=12)

plt.show()

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Split dataset into 80% train, 20% test
#X_train, X_test, y_train, y_test= train_test_split(df[['Hits','Years']], df['salary'],test_size=0.2,random_state=1706)
X_train=df.loc[:,['Hits','Years']]
y_train=df.loc[:,'salary']
# Instantiate dt
# Interactive: change max_depth to 3, 4, or None — watch in-sample MSE and the number of leaves.
dt = DecisionTreeRegressor(max_depth=2, random_state=1706)
dt.fit(X_train,y_train)

# Predict test set labels
y_pred = dt.predict(X_train)
# Evaluate test-set accuracy
print(f"In-sample MSE: {mean_squared_error(df['salary'], y_pred)}")

#store label name
labels = X_train.columns.tolist()

# Plot decision tree (matplotlib; no graphviz package needed)
fig, ax = plt.subplots(figsize=(11, 7))
plot_tree(dt, feature_names=labels, filled=True, rounded=True, fontsize=9, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
df['Salary_Predicted']=y_pred
# Split on Years
df.loc[df.Years<=4.5,'Year_Split']='Years $\leq$ 4.5'
df.loc[df.Years>4.5,'Year_Split']='Years $>$ 4.5'
# If Split on Years is True
df.loc[((df.Years<=4.5) & (df.Hits<=15.5)),'Hits_Split']=r"Hits $\leq$ 15.5"
df.loc[((df.Years<=4.5)& (df.Hits>15.5)),'Hits_Split']=r'Hits $>$ 15.5'
# If Split on Years is False
df.loc[((df.Years>4.5) & (df.Hits<=117.5)),'Hits_Split']=r"Hits $\leq$ 117.5"
df.loc[((df.Years>4.5)& (df.Hits>117.5)),'Hits_Split']='Hits $>$ 117.5'
df['Path']= df['Year_Split'] +' and '+df['Hits_Split']
df.head()

In [ ]:
sns.set_palette("Set2")
fig, ax=plt.subplots(1,1, figsize=(12,12))

sns.scatterplot(data=df, x='Years', y='Hits',hue='Path', size='Salary')

# single split on year at 4.5
ax.axvline(x=4.5, color='k', linestyle=':')

# Years <4.5 and Hits <=15.5
ax.plot((0, 4.5), (15.5, 15.5), 'k:')
# Years >4.5 and Hits <=117.5
ax.plot((4.5, 25), (117.5, 117.5), 'k:')

ax.set_xlabel("YEARS", fontsize=20)
ax.set_ylabel("HITS", fontsize=20)

ax.set_xlim(xmin=0,xmax=25)
plt.text(1.5, 7, f"R1: {str(round(np.exp(7.243499)))}", horizontalalignment='left', size='large', color='black', weight='semibold')
plt.text(1.5, 190, f"R2: {str(round(np.exp(5.058228)))}", horizontalalignment='left', size='large', color='black', weight='semibold')
plt.text(15, 7, f"R3: {str(round(np.exp(5.998380)))}", horizontalalignment='left', size='large', color='black', weight='semibold')
plt.text(15, 190, f"R4: {str(round(np.exp(6.739687)))}", horizontalalignment='left', size='large', color='black', weight='semibold')

plt.title("Splits and Predicted Salaries",fontsize=24)
plt.show()

## How the fitted tree relates to the plots

- The scatter plot with split lines shows the **partition** of the $(\text{Years}, \text{Hits})$ plane into four **regions** $R_1,\ldots,R_4$.
- The tree diagram lists the **same** splits in order: first **Years**, then **Hits** (possibly at different thresholds in different branches).
- Predictions are **constant within each region** (mean log salary); that matches the story that **rookies** vs. **veterans** and **low vs. high** hit rates matter in different ways.

**Try it (after class or live):** In the fitting cell, change `max_depth` from `2` to `3` or `4`. How many regions do you get? Does in-sample error go down?


In [ ]:
df.groupby('Path').agg({'salary':np.mean,'Salary_Predicted':np.mean})

## Terminology
- R1, R2, R3 and R4 are known as __terminal nodes__ or __leaves__
    - Trees are typically drawn upside down, in the sense that the leaves are at the bottom of the tree. 
- The points along the tree where the predictor space is split are referred to as __internal nodes__. 
    - This is where the splits occur
- We refer to the segments of the trees that connect the nodes as __branches__.

# In-Class Exercise #1

Q1: In words, what is a **terminal node (leaf)** in a regression tree? What prediction is made for every observation that falls into the same leaf?


## Interpreting the results

Read the tree **from the top down** (first split = most important partition for reducing RSS in this greedy fit):
- **Years** is the primary split: less experienced players are separated from more experienced players.
- Among **less** experienced players, **Hits** barely moves predicted salary (flat split in that slice—possibly a few unusual points).
- Among **more** experienced players, **Hits** matters more for salary: more hits $\to$ higher predicted salary in that branch.

You *could* approximate this with a linear regression with interactions, but the tree gives a **transparent** map from $(\text{Years}, \text{Hits})$ to predicted salary.


# In-Class Exercise #2

Q2: At each split, the regression tree picks a variable and a cutpoint to **reduce RSS**. Why might the **first** split be on *Years* rather than *Hits*?

Q3: In a leaf, the fitted value is the mean of **log(Salary)**. How would you convert a predicted log salary back to **thousands of dollars**?

Q4: If you increase `max_depth`, what usually happens to **training** MSE? What might happen to **test** MSE if depth is too large?


# Decision trees: under the hood
- What we did here is predict salary using __stratification of the feature space__ which we did in 2 steps:
    1. We divide the predictor space—that is, the set of possible values for $X_1,X_2, . . .,X_p$—into J distinct and non-overlapping regions, $R_1,R_2, \dots , R_J$
        - In our example, J=4
    2. For every observation that falls into the region $R_j$, we make the same prediction, which is simply the mean of the response values for the training observations in $R_j$

## Dividing the predictor space
How do we construct the regions $R_1, \dots,R_J$? 
- We divide the predictor space into high-dimensional rectangles, or __boxes__
- The goal is to find boxes R1, . . . , RJ that minimize the RSS!
    $$\Large \sum_{j=1}^J \sum_{i \in R_j} (y_i - \hat{y}_{R_j})^2$$
- Here $\hat{y}_{R_j}$ is __the average__ of $y$ in the $j$-th region $R_j$
For J=2:
$$  \sum_{i \in R_1} (y_i - \hat{y}_{R_1})^2 +\sum_{i \in R_2} (y_i - \hat{y}_{R_2})^2 $$
 

## Dividing the predictor space continued
- Note that we are no longer trying to fit data using parameters ($\beta$) 
- Our prediction for y instead relies on simply taking the average of the group j
- This is a method known as __nonparametric__ estimation
- As you can imagine, the question is how to best minimize our objective function (RSS)

## Dividing the predictor space, still
- Consider the _Years_ predictor: the cutpoint 4.5 is the value that minimizes RSS
- It means we divided the predictor space into two regions $\left\{X|X_j \leq s\right\}$ and $\left\{X|X_j > s\right\}$ such that the cutpoint $s$ leads to the greatest possible reduction in RSS. 
    - The notation means the region of predictor space in which $X_j$ is less than or equal to $s$ (or greater than $s$).

In [ ]:
dd=df.copy()
cutpoints=np.linspace(df['Years'].min(),df['Years'].max(),((df['Years'].max()-df['Years'].min())*2)+1)
RSS=[] ; cutoffs=[]
for s in cutpoints:
    # split the data along the cutoff
    dd_inf=dd.loc[df.Years<=s].copy() ; dd_sup=dd.loc[df.Years>s].copy()
    dd_inf['y_hat']=dd_inf['salary'].mean() ; dd_sup['y_hat']=dd_sup['salary'].mean()
    # take the square of the residual of each region
    dd_inf['residual_sq'] = np.square(dd_inf['salary'] - dd_inf['y_hat'] ) 
    dd_sup['residual_sq'] = np.square(dd_sup['salary'] - dd_sup['y_hat'] ) 
    # take the sum of the squared residual
    rss_inf=dd_inf['residual_sq'].sum(); rss_sup=dd_sup['residual_sq'].sum()
    # add the RSS of each region and append in RSS list
    RSS.append(rss_inf+rss_sup)  ; cutoffs.append(s)
    
dd=pd.DataFrame({'RSS':RSS, 'Cutpoint':cutoffs})  
dd.head()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(12,8))
sns.lineplot(x='Cutpoint',y='RSS',color="darkorange",data=dd,ax=ax)
ax.axvline(4.5,color='darkgreen')
ax.set_xlabel("Value of Split for Years", fontsize=20)
ax.set_ylabel("Residual Sum of Squares from split of predictor space", fontsize=16)

plt.show()

## Dividing the predictor space, next

- So we now know that we should split the data along Years=4.5
- Next, we repeat the process, looking for the best predictor and best cutpoint in order to split the data further so as to minimize the RSS within each of the __resulting regions__. 
    - However, this time, instead of splitting the entire predictor space, we split one of the two previously identified regions.
- The process continues until a stopping criterion is reached; 
    - for instance, we may continue until no region contains more than five observations.
- Once the regions $R_1, \dots , R_J$ have been created, we predict the response for a given test observation using the mean of the training observations in the region to which that test observation belongs

# Part 2: Classification trees

So far the outcome was **numeric** (log salary). In many applications the outcome is **categorical** (e.g., disease yes/no, default yes/no). The tree structure is the same: splits partition $X$-space; the prediction in each leaf is **not** a mean but a **class**—typically the **majority class** among training observations in that leaf.

**Impurity:** We cannot use RSS across classes. Instead, each candidate split is scored using **classification error**, **Gini index**, or **entropy** (sklearn's `criterion`). Gini and entropy are usually preferred because they are more sensitive for **growing** the tree.


## Error rate
- In the classification setting, RSS cannot be used as a criterion for making the binary splits.
- Instead we can use the __classification error rate__. 
    - the fraction of the training observations in that region that do not belong to the most common class:
$$ \text{Error Rate}= 1 - \max_k(\hat{p}_{mk}) $$
- Here $\hat{p}_{mk}$ represents the proportion of training observations in the $m_{th}$ region that are from the $k^{th}$ class. 
    - However, it turns out that the classification error rate is not sufficiently sensitive for tree-growing, and in practice two other measures are preferable.

## Gini Index
$$\text{Gini}=\sum_{k=1}^K \hat{p}_{mk}(1-\hat{p}_{mk})$$
- Gini is a measure of total variance across the K classes. 
    - Gini index takes on a small value if all of the $\hat{p}_{mk}$ are close to zero or one. 
- Gini index is a measure of __node purity__ 
    - a small value indicates that a node contains predominantly observations from a single class.

## Entropy
$$\text{Entropy}= - \sum_{k=1}^K \hat{p}_{mk}\log\hat{p}_{mk}$$
- Since $0 \leq \hat{p}_{mk} \leq 1$ , it follows that $0 \leq −\hat{p}_{mk}\log\hat{p}_{mk}$. 
- Entropy will take on a value near zero if the $\hat{p}_{mk}$ are all near zero or near one. 
- Gini or entropy are the preferred ways to measure the quality of a particular split

# In-Class Exercise #3

Q5: For **classification** trees, why do we **not** use RSS to choose splits?

Q6: What does the tree predict in a leaf when the outcome is **binary** (e.g. AHD yes/no)?


In [ ]:
df2=pd.read_csv('https://www.dropbox.com/scl/fi/rbcz8o1ae6a6kza05b2bl/Heart.csv?rlkey=6fqbcmkzx5gp60i0orty9vf7o&dl=1')
df2.pop(df2.columns[0])
display(df2.head())
df2['ChestPain'] = pd.factorize(df2.ChestPain)[0]
df2['Thal'] = pd.factorize(df2.Thal)[0]
df2['AHD']=df2['AHD'].replace({"Yes":1, "No":0})
df2.dropna(inplace=True)
y_train2=df2.pop('AHD') 
X_train2=df2
df2.head()

### Classification tree (Gini impurity)

The next cell fits a tree with `criterion='gini'` (sklearn's default for `DecisionTreeClassifier`). Compare the **shape** of the tree to the regression tree earlier.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
clf = DecisionTreeClassifier(max_depth=None, max_leaf_nodes=6, max_features=3, random_state=1706)
clf.fit(X_train2,y_train2)
display(clf.score(X_train2,y_train2))

#store label name
labels = X_train2.columns.tolist()

print("Classification tree using Gini")
fig, ax = plt.subplots(figsize=(14, 8))
plot_tree(clf, feature_names=labels, class_names=['No', 'Yes'], filled=True, rounded=True, fontsize=7, ax=ax)
plt.tight_layout()
plt.show()

### Classification tree (entropy)

Same settings except `criterion='entropy'`. For many datasets the fitted tree is **similar**; differences show up in which split is marginally preferred when Gini and entropy disagree.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
clf = DecisionTreeClassifier(max_depth=None, max_leaf_nodes=6, max_features=3, criterion="entropy", random_state=1706)
clf.fit(X_train2,y_train2)
display(clf.score(X_train2,y_train2))

#store label name
labels = X_train2.columns.tolist()

print("Classification tree using entropy")
fig, ax = plt.subplots(figsize=(14, 8))
plot_tree(clf, feature_names=labels, class_names=['No', 'Yes'], filled=True, rounded=True, fontsize=7, ax=ax)
plt.tight_layout()
plt.show()

### Linear models vs. regression trees (two ways to write $f$)

**Linear regression** (with vector $X = (X_1,\ldots,X_p)'$):
$$y = f(X) = \beta_0 + \sum_{j=1}^p \beta_j X_j.$$
The surface is a **hyperplane** (line if $p=1$); interactions require you to **build** them (e.g. $X_1 X_2$).

**Regression tree** (regions $R_1,\ldots,R_M$ partition the predictor space):
$$y = f(X) = \sum_{m=1}^M c_m \, \mathbf{1}_{\{X \in R_m\}}.$$
The fitted function is **piecewise constant**: $c_m$ is the mean of $y$ in region $m$. **Interactions** between predictors appear automatically when splits involve different variables on different branches.

**When might each win?** If the truth is close to linear and $n/p$ is healthy, OLS or regularized linear models often **generalize** well and are **stable**. If the relationship is **strongly nonlinear** or **local** (different behavior in different parts of $X$-space), trees (and ensembles) can **fit** better—but a single tree can be **high variance**.


# In-Class Exercise #4

Q7: Write the regression tree model as **piecewise constant** on regions $R_1,\ldots,R_M$. How is that different from a **linear** model $y = \beta_0 + \sum_j \beta_j X_j$?

Q8: Give **one** setting where you might prefer a **linear** model for prediction, and **one** where a **tree** might be more attractive (interpretability or flexibility).


**Figure (schematic):** Linear decision boundary vs. **axis-aligned** partition of the feature space typical of a tree.

![](figure8_7.png)

*Trees can approximate nonlinear boundaries by stacking many small rectangles; a linear model needs explicit nonlinear terms or kernels.*


## Trees, Pros and Cons

| Pros | Cons |
| --- | --- |
|- Trees are very easy to explain to people, including non-experts | - Trees generally do not have the same level of predictive accuracy as some of the other regression and classification approaches seen in this class |
|- Close to human decision-making (cons?) | - Trees can be very non-robust. In other words, a small change in the data can cause a large change in the final estimated tree|
|- Trees can easily handle qualitative predictors without the need to create dummy variables | |


# Part 3: Bagging, random forests, and boosting

Deep trees have **low bias** but **high variance**: small changes in the training data can change splits a lot. **Ensemble methods** combine many trees to get better **prediction** at the cost of some interpretability.

| Method | One-sentence idea |
| --- | --- |
| **Bagging** | Fit many trees on bootstrap samples; **average** predictions (regression) or **vote** (classification). Variance drops because averaged random trees are less noisy. |
| **Random forest** | Like bagging, but each split only sees a **random subset** of predictors ($m < p$). Trees become **less correlated**, so averaging helps more. |
| **Boosting** | Trees are fit **sequentially** to **residuals** (or pseudo-residuals); each new tree corrects mistakes. **Slow** learning (small step size) reduces overfitting. |

We unpack each below, then apply them in the **Boston** case study.


## Bagging (bootstrap aggregation)

- Single deep trees often have **high variance**: if you split the training data at random and fit a tree to each half, the two trees can look **very** different. Linear regression, by contrast, often has **lower variance** when $n$ is reasonably large relative to $p$.
- **Bagging** applies to many learners, but it is especially helpful for **high-variance, low-bias** base learners such as **deep trees**.
- **Bootstrap aggregation** means: draw many bootstrap samples from the training data, fit a tree on each, then **average** predictions (regression) or take a **majority vote** (classification). That averaging is what **reduces variance**.


## Bagging, continued

- Given a set of n independent observations $Z_1, \dots , Z_n$, each with variance $\sigma^2$, the variance of the mean $\bar{Z}$ of the observations is given by $\sigma^2/n$. 
    - Averaging a set of observations reduces variance!
    
- A natural way to reduce the variance and hence increase the prediction _accuracy_ of a statistical learning method is to take many training sets from the population, build a separate prediction model using each training set, and average the resulting predictions. 
- In other words, we could calculate $\hat{f}^1(X), \hat{f}^2(X), \dots, \hat{f}^B(X)$ using B separate training sets and average them in order to obtain a single low-variance statistical learning model

## Bootstrapping in Bagging
- In reality we rarely have different training datasets
- Bootstrapping consists of taking repeated samples from the (single) training data set. 
- In this approach we generate B different bootstrapped training data sets. 
- We then train our method on the $b^{th}$ bootstrapped training set in order to get $\hat{f}^{\ast b}(X)$ and average:
$$\hat{f}_{\text{bag}}(x)= \frac{1}{B} \sum_{b=1}^B \hat{f}^{\ast b}(x)$$

## Bagging: why it works
- While bagging can improve predictions for many regression methods, it is particularly useful for decision trees. To apply bagging to regression trees 
    - we simply construct B regression trees using B bootstrapped training sets, and average the resulting predictions. 
    - These trees are grown deep, and are not pruned. 
    - Hence each individual tree has high variance, but low bias. 
- Averaging these B trees reduces the variance. 
- Bagging has been demonstrated to give impressive improvements in accuracy by combining together hundreds or even thousands of trees into a single procedure.

### Out-of-Bag Error Estimation
- There is a very straightforward way to estimate the test error of a bagged model, without the need to perform cross-validation or the validation set approach. 
- Recall that the key to bagging is that trees are repeatedly fit to bootstrapped subsets of the observations.
- On average, each bagged tree makes use of around two-thirds of the observations
- The remaining one-third of the observations not used to fit a given bagged tree are referred to as the __out-of-bag (OOB) observations__
- with B sufficiently large, OOB error is virtually equivalent to leave-one-out cross-validation error

# In-Class Exercise #5

Q9: What are **out-of-bag (OOB)** observations used for when you fit many bagged trees?


In [ ]:
df = pd.read_csv("https://www.dropbox.com/scl/fi/oru1igseedqy6ec2s8dtv/Boston.csv?rlkey=ct6n06u3lxb0p8nui7uhg4urv&dl=1")
display(df.head())
y_train = df.pop('medv')
X_train = df.copy()

In [ ]:
from sklearn.ensemble import BaggingRegressor

regr = BaggingRegressor(estimator=DecisionTreeRegressor(),oob_score=True, n_estimators=100, random_state=1706).fit(X_train, y_train)
print(regr.oob_score_, regr.score(X_train,y_train))


In [ ]:
scores=[] ;  num_B=[]
for i,B in enumerate(range(20,300)):
    regr = BaggingRegressor(estimator=DecisionTreeRegressor(),oob_score=True, n_estimators=B, random_state=1706).fit(X_train, y_train)
    # Predict test set labels
    scores.append(mean_squared_error(y_train, regr.predict(X_train))) ; num_B.append(i+1)
fig, ax = plt.subplots(1,1,figsize=(10,6))  

sns.lineplot(x=num_B, y=scores)
plt.show()     

### Variable Importance Measures
- Bagging improves prediction accuracy at the expense of interpretability since it is the average of many trees
- One can obtain an overall summary of the importance of each predictor using:
    - the RSS (for bagging regression trees)
    - the Gini index (for bagging classification trees)

In [ ]:
feature_importances = np.mean([tree.feature_importances_ for tree in regr.estimators_], axis=0)

Importance = pd.DataFrame({'Importance':feature_importances, 'Features':X_train.columns})
Importance.sort_values('Importance', axis=0, ascending=True,inplace=True)

fig, ax=plt.subplots(1,1,figsize=(12,8))
sns.barplot(y='Features', x='Importance',data=Importance, orient='h', ax=ax)
ax.set_xlabel("Total Decrease in RSS due to Feature", fontsize=20)
plt.show()

## Random Forests
- __Random forests__ provide an improvement over bagged trees by way of a small tweak that decorrelates the trees
- Each time a split in a tree is considered, a random sample of m predictors is chosen as split candidates from the full set of p predictors.
- The split is allowed to use only one of those m predictors. 
- A fresh sample of m predictors is taken at each split, and typically we choose $m\approx \sqrt{p}$

### Random forests: why they work
- In other words, the algorithm is not allowed to consider a majority of the available predictors. 

Why?
- Suppose that there is one very strong predictor in the data set, along with a number of other moderately strong predictors. 
- Then in the collection of bagged trees, most or all of the trees will use this strong predictor in the top split.
- Consequently, all of the bagged trees will look quite similar to each other.
    - Hence the predictions from the bagged trees will be __highly correlated__
- Averaging many highly correlated quantities does not lead to as large of a reduction in variance as averaging many uncorrelated quantities

### Random forests: why they work (continued)
- Because each split is forced to consider only a subset of the predictors, many splits will not consider the strong predictor
- This decorrelates the trees: on average trees are less variable and hence more reliable
- Note that bagging is a special case of random forests where $m=p$

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
print(X_train2.shape)
m = round(np.sqrt(X_train2.shape[1]))
print(m)

# Random forests: using m features
clf2 = RandomForestClassifier(max_features=m, random_state=1706)
clf2.fit(X_train2, y_train2)
mean_squared_error(y_train2, clf2.predict(X_train2))

## Boosting
- __Boosting__, is another approach for improving the predictions resulting from a decision tree. 
    - Like bagging, boosting is a general approach that can be applied to many statistical learning methods
- Recall that bagging involves creating multiple copies of the original training data set using the bootstrap
    - fitting a separate decision tree to each copy
    - then combining all of the trees in order to create a single predictive model. 
    - Each tree is built on a bootstrap data set, independent of the other trees 
- Boosting works in a similar way, except that the trees are grown sequentially: 
    - each tree is grown using information from previously grown trees. 
    - there is no bootstrap sampling; 
    - instead each tree is fit on a modified version of the original data set

## Boosting Algorithm
1. Set $\hat{f}(x) = 0$ and $r_i=y_i$ for all $i$ in the training set 
2. For $b = 1, 2, \dots,B$, repeat:
    - a) Fit a tree $\hat{f}^b$ with d splits (d+1 terminal nodes) to the training data $(X, r)$.
    - b) Update $\hat{f}$ by adding in a shrunken version of the new tree:
    $$\hat{f}(x) \leftarrow \hat{f}(x) + \lambda \hat{f}^b(x)$$
    - c) Update the residual:
    $$r_i=r_i-\lambda \hat{f}^b(x)$$
3. Output the boosted model:
$$\hat{f}(x) = \sum_{b=1}^B \lambda\hat{f}^b(x)$$

## Boosting explained
- Unlike fitting a single large decision tree to the data, which amounts to _fitting the data hard_ and potentially __overfitting__, the boosting approach instead _learns slowly_.
- We fit a tree using the current residuals, rather than the outcome $Y$, as the response.
-  We then add this new decision tree into the fitted function in order to update the residuals. 
- Each of these trees can be rather small, with just a few terminal nodes, determined by the parameter d in the algorithm.

## Hyperparameters (where ML practice meets statistics)

Early in the course, tuning often meant choosing **one** model (e.g. which $\lambda$ in ridge). With trees and ensembles, you routinely tune **several** knobs at once:

| Symbol | Role (boosting / ensembles) |
| --- | --- |
| $B$ | **Number of trees** — more trees can help until validation error flattens. |
| $d$ (or `max_depth`) | **Complexity** of each weak learner — deeper trees fit harder; risk of overfitting if too large. |
| $\lambda$ (`learning_rate`) | **Step size** — smaller $\lambda$ usually needs **more** trees but can generalize better. |
| $k$ (CV folds) | How we **estimate** out-of-sample error when choosing the above. |

These are **hyperparameters** (chosen by **validation** or CV), not coefficients $\beta$ estimated by minimizing training loss alone.


# In-Class Exercise #6


Q10: In the Boston case study you will tune **learning rate** $\lambda$ and tree **depth**. Holding other settings fixed, what is the usual **bias–variance** tradeoff when you increase depth?

Q11: *(Optional / take-home)* Re-run the case study with a different `random_state` on `train_test_split`. How much do test MSE and $R^2$ move? What does that say about **variance** of a single tree or tuned ensemble?


# Case study: Boston housing

This section is a **walkthrough**: run the cells in order (restart kernel and **Run All** if you want a clean pass).

**You will:**
- Fit a **single regression tree** and see how **depth** and **random seed** affect test error.
- Run **random forests** with `max_features = p` (bagging-style) and compare **OOB** $R^2$ to **test** $R^2$ as the number of trees grows.
- Tune **gradient boosting** (learning rate $\lambda$ and depth) and inspect **feature importance**.

**Context:** The Boston data are standard for teaching but **old and geographically narrow**; use them to practice **workflow**, not to draw real-estate policy conclusions.


In [ ]:
# load Boston data
boston_df = pd.read_csv('Boston.csv')
display(boston_df.info())
boston_df.head()

In [ ]:
# Separate predictors and target
X = boston_df.drop('medv', axis=1)
y = boston_df.medv
# Split between train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=0)

In [ ]:
# Regression Tree
from sklearn.tree import DecisionTreeRegressor, plot_tree
reg = DecisionTreeRegressor(max_depth=3)
reg.fit(X_train, y_train)
pred = reg.predict(X_test)

fig, ax = plt.subplots(figsize=(16, 10))
plot_tree(reg, feature_names=X.columns, filled=True, rounded=True, fontsize=8, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
## Regression Tree and Depth of the Tree
MSEs=[] ; Depth=[]
for d in np.arange(1,30):
    reg = DecisionTreeRegressor(max_depth=d, random_state=1706)
    reg.fit(X_train, y_train)
    pred = reg.predict(X_test)
    MSEs.append(mean_squared_error(y_test, pred)) ;  Depth.append(d)
fig, ax = plt.subplots(1,1, figsize=(8,8))
sns.lineplot(x=Depth, y=MSEs,ax=ax)
ax.set_xlabel("Depth of Tree",fontsize=16)
ax.set_ylabel("Test MSE",fontsize=16)

plt.show()

In [ ]:
## Regression Tree and Random State
MSEs=[] ; States=[]
for state in np.arange(1,300):
    reg = DecisionTreeRegressor(max_depth=10, random_state=state)
    reg.fit(X_train, y_train)
    pred = reg.predict(X_test)
    MSEs.append(mean_squared_error(y_test, pred)) ;  States.append(state)
fig, ax = plt.subplots(1,1, figsize=(8,8))
sns.histplot(x=MSEs,stat='probability',ax=ax)
ax.set_xlabel('Test MSE score',fontsize=16)
plt.show()

In [ ]:
# Bagging
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
oob_bag=[] ; r2_bag=[] # create list to record predictive performance
num_trees=[]
m=X_train.shape[1] # set m to the number of features in the data
for i,B in enumerate(range(20,300)):
    reg_bag = RandomForestRegressor(n_estimators=B, max_features=m, oob_score=True,random_state=1706).fit(X_train, y_train) # recall that bagging is a special case of random forest in which max features is the number of predictors
    oob_bag.append(reg_bag.oob_score_) ; r2_bag.append(r2_score(y_test, reg_bag.predict(X_test))) ; num_trees.append(i)
res=pd.DataFrame({'OOB Score':oob_bag, r"Test $R^2$":r2_bag, "Trees": num_trees})

In [ ]:
fig, ax= plt.subplots(1,1, figsize=(10,8))
sns.lineplot(x='Trees', y='OOB Score', color='darkorange',linestyle=":",label=r'OOB $R^2$ Score', data=res, ax=ax)
sns.lineplot(x='Trees', y='Test $R^2$', color='darkgreen',linestyle=":",label=r"Out of Sample $R^2$", data=res,ax=ax)
ax.set_ylabel(r"$R^2$", fontsize=16)
ax.set_xlabel("Number of Trees", fontsize=16)

plt.show()

In [ ]:
# Boosting
from sklearn.ensemble import GradientBoostingRegressor
B=100
learning_rates=np.linspace(0.001,1,30)
depths=np.arange(1,31)
MSEs=[] ; lambdas=[] ; depth_values=[]
for 𝜆 in learning_rates:
    for d in depths:   
        boo = GradientBoostingRegressor(n_estimators=B,max_depth=d, learning_rate=𝜆, random_state=1706).fit(X_train, y_train)
        MSEs.append(mean_squared_error(y_test,boo.predict(X_test))) ; lambdas.append(𝜆) ; depth_values.append(d)

In [ ]:
res=pd.DataFrame({'MSE':MSEs, '𝜆':lambdas, 'Depth':depth_values})
print(f" Best learning rate: {float(res.loc[res.MSE==res.MSE.min(),'𝜆'])}")
print(f" Best Depth: {int(res.loc[res.MSE==res.MSE.min(),'Depth'])}")

res.head()

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(15,8))
sns.lineplot(x=r'𝜆', y='MSE', data=res, ax=axes[0])
sns.lineplot(x='Depth', y='MSE', data=res, ax=axes[1])

axes[0].set_xlabel(r"$\lambda$ - Learning Rate", fontsize=16)
axes[1].set_xlabel("Depth", fontsize=16)

plt.show()

In [ ]:
best_learning_rate=float(res.loc[res.MSE==res.MSE.min(),'𝜆'])
best_depth=int(res.loc[res.MSE==res.MSE.min(),'Depth'])
regr = GradientBoostingRegressor(n_estimators=B, learning_rate=best_learning_rate,max_depth=best_depth, random_state=1706).fit(X_train, y_train)

feature_importance = regr.feature_importances_*100
Importance = pd.DataFrame({'Importance':feature_importance, 'Features':X_train.columns})
Importance.sort_values('Importance', axis=0, ascending=True,inplace=True)

fig, ax=plt.subplots(1,1,figsize=(12,8))
sns.barplot(y='Features', x='Importance',data=Importance, orient='h', ax=ax)
ax.set_xlabel("Total Decrease in RSS due to Feature", fontsize=20)
plt.show()

### Lecture wrap-up: what to remember

Work through **Q1–Q11** above (in exercise blocks **#1–#6**) before the problem sets.

| Idea | Takeaway |
| --- | --- |
| **Tree** | Partition $X$-space; **constant** prediction per region (mean or majority class). |
| **Greedy growth** | Each split is **locally** best; global optimum is not guaranteed. |
| **Variance** | Single trees can be **unstable**; **ensembles** trade interpretability for accuracy. |
| **RF vs boosting** | RF: parallel trees, **bagging** + random features. Boosting: **sequential**, fits **residuals**. |

**Practice:** Re-run the Boston section with a different `random_state` on the train/test split. How much do your test MSE and $R^2$ move? That instability is what ensembles are designed to tame.
